In [1]:
import cv2
import numpy as np
import joblib
import mediapipe as mp

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

In [2]:
import sys
import os

sys.path.append(os.path.abspath('..'))
from utils.features import extract_features

In [3]:
model_knn = joblib.load('../models/model_knn.pkl')
model_svm = joblib.load('../models/model_svm.pkl')
model_rf = joblib.load('../models/model_rf.pkl')
scaler = joblib.load('../models/scaler.pkl')

In [4]:
options = HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(
        model_asset_path='../hand_landmarker.task'
    ),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

landmarker = HandLandmarker.create_from_options(options)

- KNN

In [8]:
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
h, w, _ = frame.shape

fourcc = cv2.VideoWriter_fourcc(*'XVID')

out = cv2.VideoWriter(
    '../demo/knn_demo.avi',
    fourcc,
    20.0,
    (w, h)
)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    result = landmarker.detect(mp_image)
    
    if result.hand_landmarks and len(result.hand_landmarks)>0:
        hand = result.hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])
        
        h, w, _ = frame.shape
        points = []

        for lm in hand:
            cx, cy = int(lm.x * w), int(lm.y * h)
            points.append((cx, cy))

            # vẽ điểm
            cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
            
        connections = [
            (0,1),(1,2),(2,3),(3,4),
            (0,5),(5,6),(6,7),(7,8),
            (0,9),(9,10),(10,11),(11,12),
            (0,13),(13,14),(14,15),(15,16),
            (0,17),(17,18),(18,19),(19,20)
        ]

        for start, end in connections:
            cv2.line(frame, points[start], points[end], (255, 0, 0), 2)
        
        features = extract_features(coords)
        features = scaler.transform(features.reshape(1, -1))
        pred = model_knn.predict(features)[0]
        
        x_t,y_t = 10,40
        (text_w1, text_h1), _ = cv2.getTextSize(f'Model KNN:', cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(
            frame,
            (x_t - 5, y_t - text_h1 - 5),
            (x_t + text_w1 + 5, y_t + 5),
            (0,0,0),
            -1
        )
        cv2.putText(frame, f'Model KNN:', (x_t,y_t), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        
        y_t2 = y_t+text_h1+15
        (text_w2, text_h2), _ = cv2.getTextSize(f'Pred: {pred}', cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(
            frame,
            (x_t - 5, y_t2 - text_h2 - 5),
            (x_t + text_w2 + 5, y_t2 + 5),
            (0,0,0),
            -1
        )
        cv2.putText(frame, f'Pred: {pred}', (x_t,y_t2), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    out.write(frame)        
    cv2.imshow("Realtime Sign Language", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

- SVM

In [11]:
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
h, w, _ = frame.shape

fourcc = cv2.VideoWriter_fourcc(*'XVID')

out = cv2.VideoWriter(
    '../demo/svm_demo.avi',
    fourcc,
    20.0,
    (w, h)
)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    result = landmarker.detect(mp_image)
    
    if result.hand_landmarks and len(result.hand_landmarks)>0:
        hand = result.hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])
        
        h, w, _ = frame.shape
        points = []

        for lm in hand:
            cx, cy = int(lm.x * w), int(lm.y * h)
            points.append((cx, cy))

            # vẽ điểm
            cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
            
        connections = [
            (0,1),(1,2),(2,3),(3,4),
            (0,5),(5,6),(6,7),(7,8),
            (0,9),(9,10),(10,11),(11,12),
            (0,13),(13,14),(14,15),(15,16),
            (0,17),(17,18),(18,19),(19,20)
        ]

        for start, end in connections:
            cv2.line(frame, points[start], points[end], (255, 0, 0), 2)
        
        features = extract_features(coords)
        features = scaler.transform(features.reshape(1, -1))
        pred = model_svm.predict(features)[0]
        
        x_t,y_t = 10,40
        (text_w1, text_h1), _ = cv2.getTextSize(f'Model KNN:', cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(
            frame,
            (x_t - 5, y_t - text_h1 - 5),
            (x_t + text_w1 + 5, y_t + 5),
            (0,0,0),
            -1
        )
        cv2.putText(frame, f'Model KNN:', (x_t,y_t), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        
        y_t2 = y_t+text_h1+15
        (text_w2, text_h2), _ = cv2.getTextSize(f'Pred: {pred}', cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(
            frame,
            (x_t - 5, y_t2 - text_h2 - 5),
            (x_t + text_w2 + 5, y_t2 + 5),
            (0,0,0),
            -1
        )
        cv2.putText(frame, f'Pred: {pred}', (x_t,y_t2), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    out.write(frame) 
    cv2.imshow("Realtime Sign Language", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

- Random Forest

In [14]:
cap = cv2.VideoCapture(0)

ret, frame = cap.read()
h, w, _ = frame.shape

fourcc = cv2.VideoWriter_fourcc(*'XVID')

out = cv2.VideoWriter(
    '../demo/rf_demo.avi',
    fourcc,
    20.0,
    (w, h)
)

while True:
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    result = landmarker.detect(mp_image)
    
    if result.hand_landmarks and len(result.hand_landmarks)>0:
        hand = result.hand_landmarks[0]
        coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])
        
        points = []

        for lm in hand:
            cx, cy = int(lm.x * w), int(lm.y * h)
            points.append((cx, cy))

            # vẽ điểm
            cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
            
        connections = [
            (0,1),(1,2),(2,3),(3,4),
            (0,5),(5,6),(6,7),(7,8),
            (0,9),(9,10),(10,11),(11,12),
            (0,13),(13,14),(14,15),(15,16),
            (0,17),(17,18),(18,19),(19,20)
        ]

        for start, end in connections:
            cv2.line(frame, points[start], points[end], (255, 0, 0), 2)
        
        features = extract_features(coords)
        features = scaler.transform(features.reshape(1, -1))
        pred = model_rf.predict(features)[0]
        
        x_t,y_t = 10,40
        (text_w1, text_h1), _ = cv2.getTextSize(f'Model RF:', cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(
            frame,
            (x_t - 5, y_t - text_h1 - 5),
            (x_t + text_w1 + 5, y_t + 5),
            (0,0,0),
            -1
        )
        cv2.putText(frame, f'Model RF:', (x_t,y_t), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        
        y_t2 = y_t+text_h1+15
        (text_w2, text_h2), _ = cv2.getTextSize(f'Pred: {pred}', cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
        cv2.rectangle(
            frame,
            (x_t - 5, y_t2 - text_h2 - 5),
            (x_t + text_w2 + 5, y_t2 + 5),
            (0,0,0),
            -1
        )
        cv2.putText(frame, f'Pred: {pred}', (x_t,y_t2), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

    out.write(frame)
    cv2.imshow("Realtime Sign Language", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()